In [ ]:
"""
Trustpilot Scraper

This script performs the following tasks:
1. Loads initial categories from Trustpilot if the cache (SQLite DB) is empty.
2. Runs sequential scraping passes over unvisited category pages to discover new subcategories.
   If a category page yields no intra‑website links (likely due to rate limiting), an exponential backoff
   mechanism retries the page before eventually marking it as visited.
3. Presents an alphabetized list of cleaned category names for user selection (unless only one category is provided).
4. Prompts the user for filtering options. If empty, defaults are: verified=True, claimed=True, TrustScore=4.5.
5. Scrapes the selected category across all countries (using cached country data or manual country codes) and extracts up to
   three businesses per country matching the filtering criteria.
6. Business results are cached for 168 hours (7 days) to avoid unnecessary re‑scraping.
7. HTTP responses are additionally cached using requests_cache to minimize repeated requests.
8. The final CSV output contains only: Country, Rank, Output URL, Rating, and Review Count.
"""

# -----------------------------------------------------------------------------
# Install required packages (if not already installed)
# -----------------------------------------------------------------------------
%pip install -r requirements.txt

# -----------------------------------------------------------------------------
# Imports and Global Modules
# -----------------------------------------------------------------------------
import os
import time
import sqlite3
import requests
import logging
import re
import hashlib
from bs4 import BeautifulSoup
import csv

# -----------------------------------------------------------------------------
# Global Configuration
# -----------------------------------------------------------------------------
MAX_CATEGORY_ATTEMPTS = 5             # Maximum attempts to retry a category page when no links are found.
INITIAL_BACKOFF_DELAY = 1             # Initial delay for exponential backoff (in seconds)
REQUEST_TIMEOUT = 10                  # Timeout for HTTP requests (in seconds)
STEP_ONE = True                      # Do not run category scraping; use manual categories.
STEP_TWO = True                       # Use manual categories for selection.
MANUAL_CATEGORIES = ["online_pharmacy"]  # List of manual category names.
STEP_THREE = True
STEP_FOUR = True                     # If True, fetch full country names from Trustpilot; if False, use MANUAL_COUNTRIES.
MANUAL_COUNTRIES = ["AU", "US"]        # Only used when STEP_FOUR is False.
STEP_FIVE = True
STEP_SIX = True
STEP_SEVEN = True

# -----------------------------------------------------------------------------
# Setup Working Directories, Cache Paths, and Logging
# -----------------------------------------------------------------------------
SCRIPT_DIR = os.getcwd()                                   # Current working directory.
CACHE_DIR = os.path.join(SCRIPT_DIR, "cache")              # Directory for cache files and DB.
DB_PATH = os.path.join(CACHE_DIR, "trustpilot.db")         # SQLite DB file path.
os.makedirs(CACHE_DIR, exist_ok=True)                      # Ensure cache directory exists.

# Configure logging (both file and console).
LOG_FILE = os.path.join(CACHE_DIR, "scraper.log")
logging.basicConfig(filename=LOG_FILE, level=logging.INFO, format="%(asctime)s - %(message)s")

# -----------------------------------------------------------------------------
# OPTIONAL HTTP CACHING via requests_cache
# -----------------------------------------------------------------------------
try:
    import requests_cache
    # Cache GET requests for 1 hour (3600 seconds).
    requests_cache.install_cache(cache_name=os.path.join(CACHE_DIR, 'trustpilot_http_cache'),
                                 backend='sqlite', expire_after=3600)
    CACHE_ACTIVE = True
except ImportError:
    CACHE_ACTIVE = False

# -----------------------------------------------------------------------------
# HTTP Session Setup
# -----------------------------------------------------------------------------
session = requests.Session()
session.headers.update({
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/120.0.0.0 Safari/537.36")
})

# -----------------------------------------------------------------------------
# Other Global Constants
# -----------------------------------------------------------------------------
BUSINESS_CACHE_EXPIRY = 168 * 3600  # Business results are fresh for 168 hours (7 days).

# -----------------------------------------------------------------------------
# Database Initialization
# -----------------------------------------------------------------------------
def init_db():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS categories (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT UNIQUE,
            url TEXT UNIQUE,
            visited INTEGER DEFAULT 0,
            level INTEGER,
            attempts INTEGER DEFAULT 0
        )
    """)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS businesses (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            category TEXT,
            country TEXT,
            business_name TEXT,
            url TEXT,
            rating REAL,
            claimed INTEGER,
            verified INTEGER,
            review_count INTEGER DEFAULT 0,  -- Ensure review_count is included
            params_hash TEXT,
            timestamp INTEGER
        )
    """)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS countries (
            code TEXT PRIMARY KEY,
            name TEXT
        )
    """)
    conn.commit()
    conn.close()

init_db()

# -----------------------------------------------------------------------------
# Logging Helper Function
# -----------------------------------------------------------------------------
def log_message(message):
    print(message)
    logging.info(message)

# -----------------------------------------------------------------------------
# Category Name Cleaning and Validity Functions
# -----------------------------------------------------------------------------
def clean_category_name(name):
    return re.sub(r'\s*\d+\s*$', '', name).strip()

def is_valid_category(name):
    return bool(name) and not name.isdigit()

# -----------------------------------------------------------------------------
# Database Helper Functions for Categories
# -----------------------------------------------------------------------------
def get_all_categories():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT name FROM categories")
    cats = {row[0] for row in cur.fetchall()}
    conn.close()
    return cats

def get_all_categories_flat():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT name, url FROM categories")
    cats = cur.fetchall()
    conn.close()
    valid_cats = [(name, url) for name, url in cats if is_valid_category(name)]
    return sorted(valid_cats, key=lambda x: x[0].lower())

def get_unvisited_categories(level):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT name, url FROM categories WHERE visited = 0 AND level = ?", (level,))
    categories = cur.fetchall()
    conn.close()
    return categories

def mark_category_visited(name):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("UPDATE categories SET visited = 1 WHERE name = ?", (name,))
    conn.commit()
    conn.close()

def save_category(name, url, level, known_categories):
    cleaned_name = clean_category_name(name)
    if not is_valid_category(cleaned_name) or cleaned_name in known_categories:
        return
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("INSERT OR IGNORE INTO categories (name, url, visited, level) VALUES (?, ?, 0, ?)",
                (cleaned_name, url, level))
    conn.commit()
    conn.close()
    known_categories.add(cleaned_name)

def get_visited_counts():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM categories WHERE visited = 1")
    visited = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM categories")
    total = cur.fetchone()[0]
    conn.close()
    return visited, total

def increment_attempts(name):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("UPDATE categories SET attempts = attempts + 1 WHERE name = ?", (name,))
    conn.commit()
    cur.execute("SELECT attempts FROM categories WHERE name = ?", (name,))
    attempts = cur.fetchone()[0]
    conn.close()
    return attempts

# -----------------------------------------------------------------------------
# (Not used in manual mode) Category Scraping Functions
# -----------------------------------------------------------------------------
def initialize_categories():
    if get_all_categories_flat():
        return
    log_message("🔄 No categories found in cache. Fetching initial Level 1 categories from Trustpilot...")
    try:
        response = session.get("https://www.trustpilot.com/categories", timeout=REQUEST_TIMEOUT)
    except Exception as e:
        log_message(f"⚠ Error fetching initial categories: {e}")
        return
    if response.status_code != 200:
        log_message(f"⚠ Failed to fetch initial categories, status code: {response.status_code}")
        return
    soup = BeautifulSoup(response.text, "html.parser")
    category_links = soup.select("a[href^='/categories/']")
    known_categories = get_all_categories()
    for link in category_links:
        raw_name = link.text.strip()
        url = f"https://www.trustpilot.com{link['href']}"
        save_category(raw_name, url, level=1, known_categories=known_categories)
    log_message("✅ Initial Level 1 categories discovered and saved.")

def scrape_category(name, url, level, known_categories):
    attempts = 0
    delay = INITIAL_BACKOFF_DELAY
    found_categories = set()
    visited_categories = set()
    while attempts < MAX_CATEGORY_ATTEMPTS:
        log_message(f"\n📂 Opening category: {name} (Attempt {attempts+1})")
        try:
            response = session.get(url, timeout=REQUEST_TIMEOUT)
        except Exception as e:
            log_message(f"⚠ Error fetching {url}: {e}")
            attempts += 1
            time.sleep(delay)
            delay *= 2
            continue
        if response.status_code != 200:
            log_message(f"⚠ Failed to fetch {name} (status code {response.status_code})")
            attempts += 1
            time.sleep(delay)
            delay *= 2
            continue
        soup = BeautifulSoup(response.text, "html.parser")
        sub_links = soup.select("a[href^='/categories/']")
        found_categories = set()
        visited_categories = set()
        for sub in sub_links:
            raw_sub_name = sub.text.strip()
            cleaned_sub_name = clean_category_name(raw_sub_name)
            if not is_valid_category(cleaned_sub_name):
                continue
            sub_url = f"https://www.trustpilot.com{sub['href']}"
            if cleaned_sub_name not in known_categories:
                save_category(raw_sub_name, sub_url, level + 1, known_categories)
                found_categories.add(cleaned_sub_name)
            else:
                visited_categories.add(cleaned_sub_name)
        if found_categories or visited_categories:
            log_message(f"✅ Category {name} yielded {len(found_categories)} new and {len(visited_categories)} known subcategories.")
            break
        else:
            attempts += 1
            log_message(f"⚠ No subcategories found for {name}. Likely rate limited. Retrying in {delay} seconds...")
            time.sleep(delay)
            delay *= 2
    mark_category_visited(name)
    return name, found_categories, visited_categories

def run_category_scraping_run():
    known_categories = get_all_categories()
    for level in range(1, 3):
        unvisited = get_unvisited_categories(level)
        if not unvisited:
            log_message(f"✅ No unvisited Level {level} categories in this run.")
            continue
        log_message(f"\n🔄 Fetching Level {level + 1} subcategories sequentially...")
        for name, url in unvisited:
            cat_name, found, visited = scrape_category(name, url, level, known_categories)
            v_count, t_count = get_visited_counts()
            progress = (v_count / t_count * 100) if t_count > 0 else 0
            log_message(f"✅ {cat_name} - Found {len(found)} new, {len(visited)} visited subcategories.")
            log_message(f"   Overall Progress: {v_count}/{t_count} categories visited ({progress:.1f}%)")
            for subcat in found:
                log_message(f"   ➡ {subcat}")
            time.sleep(5)

def scrape_categories_main():
    initialize_categories()
    for run in range(3):
        log_message(f"\n🚀 Starting scraping run {run + 1} for categories...")
        run_category_scraping_run()
        time.sleep(5)

# -----------------------------------------------------------------------------
# Modified Country Caching Function
# -----------------------------------------------------------------------------
def get_countries(source_url=None):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT code, name FROM countries")
    countries = cur.fetchall()
    if countries:
        conn.close()
        return countries
    if source_url is None:
        url = "https://www.trustpilot.com/"
    else:
        url = source_url
    log_message(f"🔄 Fetching list of countries from {url} ...")
    try:
        response = session.get(url, timeout=REQUEST_TIMEOUT)
    except Exception as e:
        log_message(f"⚠ Error fetching page {url}: {e}")
        conn.close()
        return []
    if response.status_code != 200:
        log_message(f"⚠ Failed to fetch page {url} (status code: {response.status_code})")
        conn.close()
        return []
    soup = BeautifulSoup(response.text, "html.parser")
    select = soup.select_one("select[data-country-selector-filter-select='true']")
    if not select:
        log_message("⚠ Could not find country selector on the page.")
        conn.close()
        return []
    options = select.find_all("option")
    countries = []
    for option in options:
        code = option.get("value", "").strip()
        name = option.text.strip()
        if code and name:
            countries.append((code, name))
            cur.execute("INSERT OR IGNORE INTO countries (code, name) VALUES (?, ?)", (code, name))
    conn.commit()
    conn.close()
    log_message(f"✅ Cached {len(countries)} countries.")
    return countries

# -----------------------------------------------------------------------------
# Business Caching Helper Functions
# -----------------------------------------------------------------------------
def is_business_cache_valid(category, param_hash, country):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("""
        SELECT MAX(timestamp) FROM businesses
        WHERE category = ? AND params_hash = ? AND country = ?
    """, (category, param_hash, str(country)))
    last_scraped = cur.fetchone()[0]
    conn.close()
    if last_scraped is None:
        return False
    return (time.time() - last_scraped) < BUSINESS_CACHE_EXPIRY

def save_businesses(category, country, businesses, param_hash):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    for biz in businesses:
        cur.execute("""
            INSERT INTO businesses (category, country, business_name, url, rating, claimed, verified, review_count, params_hash, timestamp)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (category, str(country), biz["name"], biz["url"], biz["rating"],
              biz["claimed"], biz["verified"], biz["review_count"], param_hash, int(time.time())))

    conn.commit()
    conn.close()

def mark_query_failed(category, country, param_hash):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO businesses (category, country, business_name, url, rating, claimed, verified, params_hash, timestamp)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (category, str(country), "__FAILED__", "", 0.0, 0, 0, param_hash, int(time.time())))
    conn.commit()
    conn.close()

def get_cached_businesses(category, param_hash, country):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("""
        SELECT business_name, url, rating, claimed, verified, review_count, timestamp
        FROM businesses
        WHERE category = ? AND params_hash = ? AND country = ?
          AND (? - timestamp) < ?
    """, (category, param_hash, str(country), int(time.time()), BUSINESS_CACHE_EXPIRY))
    rows = cur.fetchall()
    conn.close()
    businesses = []
    for row in rows:
        businesses.append({
            "name": row[0],
            "url": row[1],
            "rating": row[2],
            "claimed": bool(row[3]),
            "verified": bool(row[4]),
            "review_count": row[5],  # Ensure review_count is correctly retrieved
            "country": country
        })
    return businesses

# -----------------------------------------------------------------------------
#  Business Scraping per Country
# -----------------------------------------------------------------------------
def scrape_top_businesses_for_country(category_name, base_category_url, country, verified, claimed, min_trustscore):
    url = f"{base_category_url}?country={country}&sort=reviews_count"
    if min_trustscore is not None:
        url += f"&trustscore={min_trustscore}"
    if verified:
        url += "&verified=true"
    if claimed:
        url += "&claimed=true"

    param_str = f"{category_name}-{country}-{verified}-{claimed}-{min_trustscore if min_trustscore is not None else 'ANY'}"
    param_hash = hashlib.md5(param_str.encode()).hexdigest()

    if is_business_cache_valid(category_name, param_hash, country):
        log_message(f"🔄 Using cached results for {category_name} in {country} (verified={verified}, claimed={claimed}, min_trustscore={min_trustscore})")
        return get_cached_businesses(category_name, param_hash, country)

    log_message(f"\n🔍 Scraping top businesses for: {category_name} in {country} with URL: {url}")

    try:
        response = session.get(url, timeout=REQUEST_TIMEOUT)
    except Exception as e:
        log_message(f"⚠ Error fetching {url}: {e}")
        return []

    if response.status_code == 403:
        log_message(f"⚠ Rate limited (403) for {category_name} in {country}. Change your IP or retry later.")
        raise RuntimeError("Rate limit hit (403). Execution halted.")
    elif response.status_code == 404:
        mark_query_failed(category_name, country, param_hash)
        log_message(f"⚠ Received 404 for {category_name} in {country}. Marking as failed (cached for 168 hours).")
        return []
    elif response.status_code != 200:
        log_message(f"⚠ Failed to fetch businesses for {category_name} in {country} (status code {response.status_code})")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    url_tags = soup.find_all("p", class_=lambda c: c and "styles_websiteUrlDisplayed__lSw1A" in c)
    rating_tags = soup.find_all("p", class_=lambda c: c and "styles_ratingText__A2dmB" in c)

    businesses_raw = []
    for url_tag, rating_tag in zip(url_tags, rating_tags):
        biz_text = url_tag.get_text(strip=True)
        biz_url = f"https://{biz_text}" if not biz_text.startswith("http") else biz_text
        biz_name = biz_text

        rating_text = rating_tag.get_text(" ", strip=True)
        rating_match = re.search(r"TrustScore\s*([\d\.]+)", rating_text)
        rating = float(rating_match.group(1)) if rating_match else 0.0

        # Extract review count correctly
        parts = rating_text.split("|")
        if len(parts) > 1:
            review_count_str = parts[1].strip().split()[0]
            try:
                review_count = int(review_count_str.replace(",", ""))
            except ValueError:
                review_count = 0
        else:
            review_count = 0

        businesses_raw.append({
            "name": biz_name,
            "url": biz_url,
            "rating": rating,
            "review_count": review_count,  # Ensure this is stored correctly
            "claimed": claimed,
            "verified": verified,
            "country": country
        })

    sorted_businesses = sorted(businesses_raw, key=lambda biz: biz["review_count"], reverse=True)
    businesses = sorted_businesses[:3]  # Limit to top 3 per country

    if businesses:
        save_businesses(category_name, country, businesses, param_hash)
        log_message(f"✅ Scraped {len(businesses)} businesses for {category_name} in {country} (verified={verified}, claimed={claimed}, min_trustscore={min_trustscore})")
    else:
        log_message(f"ℹ️ No businesses found for {category_name} in {country} matching criteria.")

    return businesses

# -----------------------------------------------------------------------------
# Main Program Flow
# -----------------------------------------------------------------------------
def main():
    # STEP 1: (Category scraping) – Not used in manual mode.
    if STEP_ONE:
        scrape_categories_main()
    
    # STEP 2: Category selection.
    if STEP_TWO:
        # When STEP_TWO is True, we ignore MANUAL_CATEGORIES.
        cats = get_all_categories_flat()
        if not cats:
            log_message("⚠ No categories found in the database. Exiting.")
            return
        if len(cats) == 1:
            selected_category, selected_url = cats[0]
            log_message(f"Only one category available. Selected Category: {selected_category} ({selected_url})")
        else:
            print("\nAvailable Categories:")
            for idx, (name, url) in enumerate(cats, start=1):
                print(f"{idx}. {name}")
            try:
                choice = int(input("Enter the number of the category you want to scrape: ").strip())
                if choice < 1 or choice > len(cats):
                    print("Invalid choice.")
                    return
            except ValueError:
                print("Invalid input.")
                return
            selected_category, selected_url = cats[choice - 1]
            log_message(f"Selected Category: {selected_category} ({selected_url})")
    else:
        # When STEP_TWO is False, use only MANUAL_CATEGORIES.
        if not MANUAL_CATEGORIES:
            log_message("⚠ No manual categories provided. Exiting.")
            return
        # If multiple categories are provided in MANUAL_CATEGORIES, stop and ask the user to select only one.
        if len(MANUAL_CATEGORIES) > 1:
            log_message(f"⚠ Multiple manual categories provided: {MANUAL_CATEGORIES}")
            log_message("❌ Please provide only **one** category in MANUAL_CATEGORIES.")
            return
        selected_category, selected_url = MANUAL_CATEGORIES[0], f"https://www.trustpilot.com/categories/{MANUAL_CATEGORIES[0]}"
        log_message(f"Using manual category: {selected_category} ({selected_url})")  
    # STEP 3: Filtering options.
    if STEP_THREE:
        verified_input = input("Should businesses be verified? (y/n, default y): ").strip().lower()
        verified = True if verified_input in {"", "y", "yes"} else False
        claimed_input = input("Should businesses be claimed? (y/n, default y): ").strip().lower()
        claimed = True if claimed_input in {"", "y", "yes"} else False
        min_trustscore_input = input("Enter minimum TrustScore (3.0, 4.0, 4.5 or press Enter for default 4.5): ").strip()
        if min_trustscore_input == "":
            min_trustscore = 4.5
        elif min_trustscore_input in {"3.0", "4.0", "4.5"}:
            min_trustscore = float(min_trustscore_input)
        else:
            min_trustscore = 4.5

    # STEP 4: Retrieve countries.
    if STEP_FOUR:
        countries = get_countries(selected_url)
        if not countries:
            log_message("⚠ No countries available. Exiting.")
            return
    else:
        countries = [(code, code) for code in MANUAL_COUNTRIES]
    
    # STEP 5: Scrape businesses per country.
    if STEP_FIVE:
        log_message(f"\n🚀 Scraping businesses for category '{selected_category}' across {len(countries)} countries...")
        results = {}
        for country_code, _ in countries:
            attempt = 0
            max_attempts = 5
            businesses = []
            while attempt < max_attempts:
                businesses = scrape_top_businesses_for_country(selected_category, selected_url,
                                                               country_code, verified, claimed, min_trustscore)
                # If a 404 occurred, our function now returns an empty list.
                if businesses:
                    break
                else:
                    log_message(f"Country {country_code} returned no results. Not retrying further.")
                    break
            results[country_code] = businesses
    
    # STEP 6: Display results.
    if STEP_SIX:
        print("\nScraping Results:")
        for country_code, businesses in results.items():
            # When STEP_FOUR is False, use the two-letter code as the country display.
            country_display = country_code
            print(f"\nCountry: {country_display} ({country_code})")
            if businesses:
                for biz in businesses:
                    print(f" - {biz['name']} (Rating: {biz['rating']}, Claimed: {biz['claimed']}, Verified: {biz['verified']})")
            else:
                print(" No matching businesses found.")
        log_message("\n🚀 Business Scraping complete.")
    # STEP 7: Export results to CSV.
    if STEP_SEVEN:
        if not results:
            log_message("⚠ 'results' is empty. No CSV data to save.")
        else:
            csv_path = os.path.join("results.csv")
            
            # Step 1: Write initial CSV file
            with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
                writer = csv.writer(csvfile)
                writer.writerow(["Country", "Rank", "Output URL", "Rating", "Review Count"])
                for country_code, businesses in results.items():
                    valid_businesses = [biz for biz in businesses if biz.get("name")]
                    if not valid_businesses:
                        continue
                    valid_businesses = sorted(valid_businesses, key=lambda biz: biz["review_count"], reverse=True)
                    for rank, biz in enumerate(valid_businesses, start=1):
                        writer.writerow([
                            country_code,
                            rank,
                            biz.get("url", ""),
                            biz.get("rating", ""),
                            biz.get("review_count", "")
                        ])

            log_message(f"✅ Results saved to {csv_path}")

            # Step 2: Read and sort CSV data
            sorted_data = []
            with open(csv_path, "r", newline="", encoding="utf-8") as csvfile:
                reader = csv.reader(csvfile)
                header = next(reader)  # Read the header
                sorted_data = sorted(reader, key=lambda row: int(row[-1] or 0), reverse=True)  # Sort by Review Count (last column)

            # Step 3: Write back the sorted data
            with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
                writer = csv.writer(csvfile)
                writer.writerow(header)  # Write the header first
                writer.writerows(sorted_data)  # Write sorted data

            log_message(f"✅ Sorted results saved to {csv_path}")

if __name__ == '__main__':
    main()
